# BUSI lesion segmentation — Colab training notebook

**Research demo, not for clinical use.**

This notebook trains a U-Net (ResNet-18 encoder) with an auxiliary benign-vs-malignant head
on the BUSI dataset, evaluates Dice/IoU/AUC, and exports ONNX for the web demo.

Repo: https://github.com/surabhif/Breast-ultrasound-lesion-segmentation


## 1. Setup


In [ ]:
!pip -q install torch torchvision pillow scikit-learn scikit-image imagehash opencv-python-headless onnx onnxruntime tqdm pandas matplotlib scipy
import torch
print('CUDA:', torch.cuda.is_available(), 'Torch', torch.__version__)


## 2. Clone repo & download BUSI


In [ ]:
import os
from pathlib import Path
if not Path('scripts/download_busi.py').exists():
    !git clone --depth 1 https://github.com/surabhif/Breast-ultrasound-lesion-segmentation.git repo
    %cd repo
else:
    print('Already in repo root')
!python scripts/download_busi.py
!python scripts/prepare_dataset.py


## 3. Quick baseline (CPU-friendly)

Tiny U-Net at 64×64 — useful smoke test. Skip if you only want the full model.


In [ ]:
!python scripts/run_baseline_quick.py


## 4. Full training (GPU recommended)

ResNet-18 U-Net at 128×128. On Colab GPU this is much faster than CPU.


In [ ]:
!python scripts/train_full.py --epochs 20 --img-size 128 --batch-size 16 --patience 5 --run-cv --cv-epochs 3


## 5. Cleaning experiment, gallery, metrics, quantize


In [ ]:
!python scripts/run_cleaning_experiment.py
!python scripts/export_gallery.py
!python scripts/quantize_onnx.py
!python scripts/export_web_results.py
print('Artifacts ready under web/public/ and results/')


## 6. Download weights for GitHub Pages

Download `web/public/models/busi_unet.onnx` and `web/public/results/metrics.json` (plus mistakes gallery)
and commit them to the repo, or open a PR from Colab via GitHub.


In [ ]:
from google.colab import files
from pathlib import Path
for p in [
    Path('web/public/models/busi_unet.onnx'),
    Path('web/public/results/metrics.json'),
    Path('results/full_run.json'),
    Path('results/cleaning_experiment.json'),
]:
    if p.exists():
        print('download', p)
        files.download(str(p))
    else:
        print('missing', p)
